In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("ModuleofAppliedGeomorphologyfinal.pdf")
documents = loader.load()

print(len(documents))

14


Data cleanining 

In [7]:
clean_docs = []

for doc in documents:
    text = doc.page_content.replace("\n", " ")
    text = text.strip()
    clean_docs.append(text)

Chunking

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.create_documents(clean_docs)

print(len(chunks))

71


Embedding Generation (Open Source)

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Aishwarya\AppData\Local\Temp\ipykernel_16584\384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
d:\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Store in Vector Database (FAISS)

In [12]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

vector_store.save_local("faiss_index")

Retrieval

In [13]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

query = "What is the main topic of the document?"

retrieved_docs = retriever.get_relevant_documents(query)

for doc in retrieved_docs:
    print(doc.page_content)

Items  Description of Module  Subject Name Geography  Paper Name Geomorphology    Module Name/Title Applied Geomorphology    Module Id  GEO/35                                Pre-requisites Applied Geomorphology, systematic analysis and  relation with geomorphological features  Objectives To know about the application of geomorphology and  its development in different geomorphological regions  like karst, glacial, alluvial, fluvial and many more, and  its contribution in regional planning, urban
may then advise development projects best  suited for separate region.   GEOMORPHOLOGY AND URBANISATION  There is a separate branch known as urban geomorphology applied to urban development.  According to R.U. Cooke, this branch of geomorphology is concerned with “the study of  landforms and their related processes, materials and hazards, ways that are beneficial to  planning, devel opment and management of urbaniz ed areas where urban growth is expected”.
Applied Geomorphology    Component-I (A

d:\Project\.venv\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Pass Context to LLM (Prompt Construction)

In [14]:
from langchain_community.llms import Ollama

llm = Ollama(model="tinyllama")

In [15]:
# Build context from retrieved documents
context = ""

for doc in retrieved_docs:
    context = context + doc.page_content + "\n\n"

# Create prompt
prompt = "Answer the question based ONLY on the context below.\n\n"
prompt = prompt + "Context:\n"
prompt = prompt + context + "\n"
prompt = prompt + "Question:\n"
prompt = prompt + query + "\n\n"
prompt = prompt + "Answer:"

Generate Answer

In [16]:
response = llm.invoke(prompt)

print(response)

The main topic of the document is applied geomorphology and its contribution to urban planning and development.
